In [0]:
%sql
-- 1. Create Time Dimension (For Trend Analysis)
CREATE OR REPLACE TABLE workspace.healthcare3.dim_date AS
SELECT 
    DISTINCT CAST(admission_date AS DATE) AS date_key,
    day(admission_date) AS day,
    month(admission_date) AS month,
    year(admission_date) AS year,
    concat('Q', quarter(admission_date)) AS quarter
FROM workspace.healthcare3.silver_patients;

-- 1. NEW: Patient Dimension (Unique on patient_key)
-- Stores static attributes to allow filtering by Age/Gender.
CREATE OR REPLACE TABLE workspace.healthcare3.dim_patient AS
SELECT 
    DISTINCT patient_key,
    gender,
    blood_group
FROM workspace.healthcare3.silver_patients;

-- 2. CMO Dimension (Clinical Profiles)
CREATE OR REPLACE TABLE workspace.healthcare3.dim_cmo AS
SELECT 
    sha2(concat(clinical_risk_segment, blood_pressure_status, lab_result_status), 256) AS cmo_key,
    clinical_risk_segment,
    blood_pressure_status,
    lab_result_status
FROM (
    SELECT DISTINCT clinical_risk_segment, blood_pressure_status, lab_result_status 
    FROM workspace.healthcare3.silver_patients
);

-- 3. Payer Dimension (Clean 1:M)
CREATE OR REPLACE TABLE workspace.healthcare3.dim_payer AS
SELECT DISTINCT payer_name AS payer_key, payer_name
FROM workspace.healthcare3.silver_patients;

-- 4. Resource Dimension (Clean 1:M)
CREATE OR REPLACE TABLE workspace.healthcare3.dim_coo AS
SELECT DISTINCT dept_name AS dept_key, dept_name
FROM workspace.healthcare2.silver_patients;

-- 5. Updated Fact Table (Now including ALL important features)
CREATE OR REPLACE TABLE workspace.healthcare3.fact_hospital_performance AS
SELECT 
    patient_key,             -- Links to dim_patient
    dept_name AS dept_key,   -- Links to dim_hospital_resource
    payer_name AS payer_key, -- Links to dim_payer
    CAST(admission_date AS DATE) AS date_key,
    sha2(concat(clinical_risk_segment, blood_pressure_status, lab_result_status), 256) AS cmo_key,
    
    -- Added Demographic Feature (Stored in fact for encounter-age accuracy)
    age AS admission_age,
    
    -- Added Clinical Vitals (CMO Deep-Dive)
    heart_rate,
    oxygen_saturation,
    comorbidity_count,
    treatment_type,
    
    -- Added Operational Efficiency (COO)
    is_emergency_admission, -- NEW: Planned vs. Unplanned impact
    los_days,
    wait_time_min,
    is_er_overload,
    bed_occupancy_status,
    staff_ratio,
    
    -- Added Financial & Cashback Metrics (CFO Pillar)
    total_billed_amount,    -- NEW: The "starting" bill
    claim_amount,
    approved_claim_amount,
    revenue_leakage,
    cashback_recovery_rate,
    claim_status,
    denial_reason
FROM workspace.healthcare3.silver_patients;

-- 6. Re-Optimize for Power BI
OPTIMIZE workspace.healthcare3.fact_hospital_performance 
ZORDER BY (date_key, payer_key, cmo_key, patient_key);
-- 6. Executive Validation Query (Proving the "Cashback" ROI)
SELECT 
    payer_key AS Payer,
    COUNT(patient_key) AS Total_Claims,
    ROUND(AVG(cashback_recovery_rate) * 100, 2) AS Recovery_Rate_Pct,
    ROUND(SUM(revenue_leakage), 2) AS Total_Leakage_USD
FROM workspace.healthcare3.fact_hospital_performance
GROUP BY 1
ORDER BY Total_Leakage_USD DESC;

Payer,Total_Claims,Recovery_Rate_Pct,Total_Leakage_USD
Medicare,60136,75.63,2.6153804789E8
BlueCross,60070,75.69,2.6098049964E8
Self-Pay,60355,75.7,2.605344702E8
Aetna,59871,75.81,2.5966277345E8
UnitedHealth,59568,75.8,2.5756930225E8
